# EDA

## Import Packages

In [1]:
import os
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

## Prepare

In [71]:
STORAGE_DIR = os.path.join("../input/")

In [73]:
OUTPUT_DIR = os.path.join("../output/")

## Load CSV-File

In [13]:
masterfile_df = pd.read_csv(os.path.join(OUTPUT_DIR, "df_masterfile.csv"))

## Inspect

In [17]:
masterfile_df.dtypes

Unnamed: 0      int64
id             object
title          object
preview        object
date           object
zeitschrift    object
dtype: object

### Säubern der Variable `date`

**NOTE:**

---

Die Variable `date` enthält eine Mischung an Wert-Formatierungen und wird zudem nicht als Datetime-Format erkannt. Daher werden wir die Werte der Variable im Abschnitt Data Cleansing/Datum formatieren.
  
---

In [19]:
masterfile_df["date"].count

<bound method Series.count of 0          Heute 15:29 Uhr
1          Heute 15:21 Uhr
2          Heute 15:16 Uhr
3          Heute 15:15 Uhr
4          Heute 15:07 Uhr
              ...         
723    2025-11-30 00:00:00
724    2025-10-30 00:00:00
725    2025-10-30 00:00:00
726    2025-10-14 00:00:00
727    2025-10-14 00:00:00
Name: date, Length: 728, dtype: object>

### Fehlende Werte in der Variable `preview`

Der Masterfile hat ingesamt 728 Fälle mit jeweils 6 Variablen

In [37]:
print(masterfile_df.shape)

(728, 6)


In der Variable `preview` fehlen 45 Werte

In [40]:
print(masterfile_df.isnull().sum())

Unnamed: 0      0
id              0
title           0
preview        45
date            0
zeitschrift     0
dtype: int64


**NOTE:**

---

Über `groupby` untersuchen wir, wo die fehlenden Werte zu finden sind. Im Anschluss entscheiden wir, ob der Anteil signifikant ist.

---

In [43]:
groupby_masterfile_df = masterfile_df.groupby(["zeitschrift"]).count()

groupby_masterfile_df = groupby_masterfile_df.drop(columns=["Unnamed: 0", "title", "date"])

groupby_masterfile_df["preview_percentage"] = groupby_masterfile_df["preview"]/groupby_masterfile_df["id"]

groupby_masterfile_df

,id,preview,preview_percentage
zeitschrift,,,
spielgel,12,12,1.000000
sz,350,305,0.871429
zeit,366,366,1.000000


**NOTE:**

---

Es stellt sich heraus, dass nur das Magazin `sz` fehlende Werte in der Variable `preview` aufweist. Auch wenn 13 % nicht besonders niedrig sind, bleiben uns dennoch 305 Beobachtungen, was mehr als ausreichend ist. Daher können wir die 45 Fälle aus unserem Datensatz entfernen.

---

## Data Cleansing

### Datum formatieren

**NOTE:**

---
Die Variable `date` muss vorerst bereinigt werden. Die Zeilen, die den Wert `⁣Heute` haben, werden mit dem vollständigen Datumsformat ersetzt.

---

In [27]:
masterfile_df["date"] = masterfile_df["date"].apply(lambda a: re.sub("^Heute.*", "2026-01-30", str((a))))

**Die Uhrzeit im Datum entfernen**

In [30]:
masterfile_df["date"] = masterfile_df["date"].str.replace(" 00:00:00","")

**Die Spalte `date` in datime formatieren**

In [33]:
masterfile_df["date"] = pd.to_datetime(masterfile_df["date"], dayfirst=True, format="mixed")

In [35]:
masterfile_df.dtypes

Unnamed: 0              int64
id                     object
title                  object
preview                object
date           datetime64[ns]
zeitschrift            object
dtype: object

In [45]:
masterfile_df.shape

(728, 6)

In [47]:
cleaned_masterfile_df = masterfile_df.dropna()

In [49]:
cleaned_masterfile_df.shape

(683, 6)

In [53]:
print(cleaned_masterfile_df.isnull().sum())

Unnamed: 0     0
id             0
title          0
preview        0
date           0
zeitschrift    0
dtype: int64


## Export Cleaned DataFrame

In [56]:
cleaned_masterfile_df.to_csv(os.path.join(OUTPUT_DIR,"cleaned_masterfile_df.csv"), sep=",")